# IFS-FESOM along-track plot — earthkit refactor

Refactor of the last four code cells of `dt-climate_IFS-FESOM_download_plot.ipynb`:

- the four copy-pasted CSV merge blocks collapse into one loader loop that
  aligns every IFS-FESOM series on the MOSAiC obs time index (Kelvin → °C at
  load time, duplicate timestamps dropped);
- file paths fall back from the original `Data/IFS-FESOM/*.csv` layout to the
  along-track CSVs present in this repo (`Temp2m_IFS_*_MOSAiC.csv`, …);
- plotting uses `earthkit.plots.Figure` instead of raw matplotlib.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
from earthkit.plots import Figure

In [ ]:
BASE = Path.home() / "Git/PISCO_notebooks"
DATA_DIR = BASE / "Data"
OBS_FILE = DATA_DIR / "mosaic-examples/MOSAiC_obs_combined_1h_v2.nc"
FIG_DIR = BASE / "Figures"
FIG_DIR.mkdir(exist_ok=True)

K0 = 273.15

COLORS = {"MOSAiC": "grey", "CARRA2": "darkorange",
          "IFS_hist": "tab:blue", "IFS_ctrl": "tab:red"}
LABELS = {"MOSAiC": "MOSAiC", "CARRA2": "CARRA2",
          "IFS_hist": "IFS-FESOM hist", "IFS_ctrl": "IFS-FESOM ctrl"}

# IFS-FESOM along-track CSVs: original dt-climate layout first,
# PISCO layout (the files present in this repo) as fallback
IFS_FILES = {
    "IFS_2t_ctrl":   (["IFS-FESOM/2t_ctrl.csv", "Temp2m_IFS_Ctrl_MOSAiC.csv"], "2t", True),
    "IFS_2t_hist":   (["IFS-FESOM/2t_hist.csv", "Temp2m_IFS_Hist_MOSAiC.csv"], "2t", True),
    "IFS_10si_ctrl": (["IFS-FESOM/10si_ctrl.csv", "WindSpeed10m_IFS_Ctrl_MOSAiC.csv"], "10si", False),
    "IFS_10si_hist": (["IFS-FESOM/10si_hist.csv", "WindSpeed10m_IFS_Hist_MOSAiC.csv"], "10si", False),
}

## Load MOSAiC obs + IFS-FESOM along-track series

In [ ]:
def load_hourly_frame(obs_file=OBS_FILE, ifs_files=IFS_FILES):
    """Hourly MOSAiC obs joined with the IFS-FESOM along-track series.

    Replaces the four copy-pasted merge blocks of the original: each CSV
    becomes a named series that aligns on the obs time index. 2t converts
    from Kelvin to deg C at load time.
    """
    obs = xr.open_dataset(obs_file)
    frame = pd.DataFrame({
        "obs_t2m": obs.temp_2m.to_series(),        # deg C
        "obs_10si": obs.wspd_vec_mean_10m.to_series(),
    })
    frame = frame[~frame.index.duplicated()]

    for name, (candidates, col, kelvin) in ifs_files.items():
        for rel in candidates:
            path = DATA_DIR / rel
            if path.exists():
                break
        else:
            raise FileNotFoundError(f"{name}: none of {candidates} in {DATA_DIR}")
        df = pd.read_csv(path, index_col=0)
        s = pd.Series(df[col].values, index=pd.to_datetime(df["time"]), name=name)
        s = s[~s.index.duplicated()]
        frame[name] = s - K0 if kelvin else s      # aligns on the time index
        print(f"{name}: {s.notna().sum()} values from {path.name}")
    return frame


def as_time_da(series):
    """Series with DatetimeIndex -> DataArray with a 'time' coord.
    earthkit.plots reads the date axis from the coordinate; a bare pandas
    Series would be plotted against integer positions."""
    return series.rename_axis("time").to_xarray()


hourly = load_hourly_frame()
hourly.describe().round(2)

## Time series (earthkit.plots)

In [ ]:
fig = Figure(rows=2, columns=1, size=(10, 8))

top = fig.add_subplot()
top.line(as_time_da(hourly["obs_t2m"]), label="MOSAiC obs (1 h)",
         color=COLORS["MOSAiC"], linewidth=0.9, alpha=0.85)
top.line(as_time_da(hourly["IFS_2t_hist"]), label="IFS_hist (1 h)",
         color=COLORS["IFS_hist"], linewidth=0.9)
top.line(as_time_da(hourly["IFS_2t_ctrl"]), label="IFS_ctrl (1 h)",
         color=COLORS["IFS_ctrl"], linewidth=0.9)
top.ax.set_title("2m air temperature [°C]")
top.ax.grid(linewidth=0.3)
top.legend()

bottom = fig.add_subplot()
bottom.line(as_time_da(hourly["obs_10si"]), label="MOSAiC obs (1 h)",
            color=COLORS["MOSAiC"], linewidth=0.9, alpha=0.85)
bottom.line(as_time_da(hourly["IFS_10si_hist"]), label="IFS_hist (1 h)",
            color=COLORS["IFS_hist"], linewidth=0.9)
bottom.line(as_time_da(hourly["IFS_10si_ctrl"]), label="IFS_ctrl (1 h)",
            color=COLORS["IFS_ctrl"], linewidth=0.9)
bottom.ax.set_title("10m wind speed [m s$^{-1}$]")
bottom.ax.grid(linewidth=0.3)
bottom.legend()

fig.save(str(FIG_DIR / "IFS-FESOM_MOSAiC_Timeseries.png"), dpi=300,
         bbox_inches="tight")
fig.show()